# Notebook 07 -- ensemble merge and inter-model agreement

**Asks:** combine the models and test whether they agree. That agreement test is
**gate clause B** of the registration (prereg 6.7: the models agree at an
intraclass correlation of at least .60).

**Reads** `llm_scores_<model>.csv`, one file per model, from notebook 06.
**Writes** `ensemble_scores.csv` (the facet score of record, long form) and
`inter_model_icc.csv` (clause B, per facet).

### What this notebook can tell you, and what it cannot

It measures whether the language models agreed **with each other**. That is
*reliability* -- consistency -- and it is not *validity* -- accuracy. Prereg 6.7
says so in as many words: "Agreement among the models is reliability, not
validity." Models trained on similar text can share a blind spot and converge,
confidently, on the same wrong reading; averaging them does not cancel that
error, it launders it into a tight consensus with a high ICC.

With the human-coder arm switched off (`HUMAN_GOLD = False`, the registered
default) there is **no independent check on shared bias anywhere in this
chain**. A high ICC here therefore licenses nothing on its own. The design
document makes the same point at section 4 and section 7, and the gate is a
union of clauses for exactly this reason: clause B (this notebook) is one of
them, never the whole of it.

## Config

In [ ]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "07"                     # this notebook's number
NB_NAME     = "ensemble_merge_and_icc" # this notebook's job, in two words
NB_ASKS     = "Combine the models and test whether they agree (gate clause B)."

## Setup

In [ ]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

In [ ]:
# === SETUP (cont.): the two packages a Colab machine does not ship ===
# WHAT THIS CELL DOES: installs the two agreement packages this notebook needs,
# then records the versions the session actually resolved. It computes nothing.
#
# WHY IT RUNS EVERY TIME: a Colab machine is thrown away when the runtime
# disconnects, so the install happens at the top of every session.
#
#   pingouin      gives us ICC(2,1) and ICC(2,k) -- the two intraclass
#                 correlations that say how far the models agree on the 0-to-1
#                 facet scores. Without it there is no clause B.
#   krippendorff  gives us Krippendorff's alpha, an agreement coefficient built
#                 for LABELS rather than numbers, which is what the holistic
#                 principled/vengeful/both/neither call is.
#
# WHAT --no-deps DOES AND WHERE IT STOPS. numpy, pandas and scipy ship in the
# Colab image compiled against one another. A pip install that resolves a
# different numpy leaves the already-built wheels pointing at the wrong C
# structures, and the next import dies with
#     ValueError: numpy.dtype size changed, may indicate binary incompatibility
# krippendorff is installed with --no-deps because numpy is its only dependency,
# so declining to resolve it costs nothing. pingouin CANNOT be installed that
# way: it needs pandas-flavor and tabulate, neither of which is in the stock
# Colab image, so the second line resolves pingouin's whole tree -- numpy,
# scipy, pandas, matplotlib, seaborn, statsmodels, scikit-learn -- and that
# resolution CAN move the compiled core. It is the one install here that can,
# which is why the versions it actually resolved are printed below and belong in
# the OSF bundle beside the results.
# IF YOU EVER SEE THAT ERROR: pip cannot swap a compiled module the live kernel
# has already imported. Runtime -> Restart session, then run from the top.
!pip install -q --no-deps krippendorff
!pip install -q pingouin
print("installed: krippendorff, pingouin")

# Reproducibility record: the resolved versions belong in the OSF bundle next to
# the results, so a later run can be compared against this one.
import importlib, sys
from importlib import metadata as _md

def _ver(mod_name, dist_name=None):
    # Not every package exposes __version__ (krippendorff does not), so fall back
    # to the installed distribution metadata before giving up.
    try:
        return importlib.import_module(mod_name).__version__
    except Exception:
        pass
    try:
        return _md.version(dist_name or mod_name)
    except Exception as e:
        return f"NOT AVAILABLE -- {type(e).__name__}"

print("\nResolved versions (archive these with the outputs):")
print(f"  python        {sys.version.split()[0]}")
for _mod in ["numpy", "pandas", "scipy", "krippendorff", "pingouin"]:
    print(f"  {_mod:13s} " + str(_ver(_mod)))

In [ ]:
# === SETUP (cont.): imports, display width, and the facet vocabulary ===
import pandas as pd
import numpy as np
import re
from collections import Counter
import warnings

# WARNINGS ARE NOT SILENCED IN THIS NOTEBOOK, and the warnings module is imported
# in order to CATCH one of them, not to switch it off. pingouin raises "divide by
# zero" and "invalid value" warnings when it is handed a facet with no variance
# to partition, and those are the runtime signal that an agreement number is
# about to be printed for something nobody measured. A blanket
# filterwarnings("ignore") -- which silences every warning for the rest of the
# session, from every library -- would hide exactly the evidence a reader needs
# in order to distrust the table. Python's own defaults are left alone; the one
# call that is expected to be noisy is wrapped in warnings.catch_warnings() and
# what it raised is REPORTED beside the facet it was raised on.

# krippendorff -> nominal alpha on the holistic principled/vengeful/both/neither
# call; pingouin -> ICC(2,1) and ICC(2,k) across models.
import krippendorff
import pingouin as pg

# The agreement table is wide (nine facets, several columns each). Without these
# two settings pandas replaces the middle columns with an ellipsis, and a
# truncated gate table is worse than no gate table.
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
print("Imports complete: pandas, numpy, krippendorff, pingouin.")

In [ ]:
# === SETUP (cont.): the facet vocabulary and the two thresholds ===
# --- The canonical facet vocabulary (prereg 5.15; design section 2) --------
# Nine multi-label facets, each scored 0.0-1.0 per response per item_type. They
# are NEVER forced to sum to 1 and never collapsed onto one axis: a response can
# be entirely consequentialist (all six retributive facets 0.0), entirely
# vengeful, or a genuine mixture. Co-occurrence is the object of measurement for
# E4, so averaging it away or subtracting one side from the other -- which the
# old prosocial-minus-dark roll-up literally did -- destroys the quantity.
#
# The order is fixed and this list is canonical: the per-model columns, the
# ensemble means, the agreement table and the long export all iterate FACETS, so
# reordering or renaming an entry here silently renames columns in two files, and
# repeating one would write the same facet twice into both.
FACETS = ["desert", "proportionality", "censure", "moral_balance",
          "revenge", "suffering", "deterrence", "incapacitation", "rehabilitation"]

# The single holistic call, deliberately kept OUT of the nine-facet vector. It
# comes from the same pass and the same rater as the facets, so it is descriptive
# coverage and a Krippendorff target -- never corroboration for facet-level
# co-occurrence, which would be one read counted twice (design section 9 rule 6).
HOLISTIC_LEVELS = ["principled", "vengeful", "both", "neither"]

# Gate clause B, in one number. Prereg 6.7: "a facet supports a claim only if the
# models agree at an intraclass correlation of at least .60". Reported per facet
# below; the pass/fail decision that matters is B together with the other clauses,
# and that union is assembled in the gate notebook, not here.
ICC_THRESHOLD = 0.60

# How many responses have to express a facet at all before an agreement number
# about it is worth reading. An ICC computed where three models agree on 895
# zeros and five positives comes back at 1.000, and that number is a fact about
# the zeros. Below this floor clause B reports NOT RUN with the count beside it.
# The floor is an addition of the measurement design and not part of prereg 6.7
# -- design section 7 imposes the same kind of floor on clause A (n_pos >= 30)
# and section 6 imposes a 5% coverage floor on the word-list leg, for the same
# reason -- so it is written into the output file and argued at the cell that
# applies it.
MIN_POSITIVE_CASES = 30

# THE TWO PRESENCE DEFINITIONS (design section 0, the presence row). The facet
# score of record is the continuous ensemble mean, and every test in this chain
# runs on it. A facet is COUNTED rather than correlated in two places only -- the
# positive-case floor above, and the prevalence lines printed beside the ICCs --
# and both counts use a definition fixed before scoring:
#   trace  at least one model scored the facet above zero. The floor counts this,
#          because the floor exists to catch an ICC built from agreed zeros, and a
#          response on which any model saw the facet is not an agreed zero.
#   clear  the ensemble mean is above PRESENT_CLEAR, i.e. the models on balance
#          called the facet present. Reported beside trace prevalence.
# DANGER: notebook 10 holds the same constant and stops if its copy differs from
# the value this notebook writes into inter_model_icc.csv. Change both or neither.
PRESENT_CLEAR = 0.50

print("Nine facets, one holistic call. Gate clause B passes at "
      f"ICC(2,k) >= {ICC_THRESHOLD:.2f} on at least {MIN_POSITIVE_CASES} "
      "responses that express the facet.")

## Load

In [ ]:
# === LOAD (1 of 4): discover the per-model score files, and count the raters ===
# WHAT THIS CELL DOES: asks the manifest which llm_scores_<model>.csv files
# notebook 06 wrote, then puts each one through need().
#
# WHY DISCOVERY AND NOT THREE HARDCODED NAMES. The roster of models is a
# MEASUREMENT decision, and it is notebook 06 that makes it. If this notebook
# named three files itself, a run in which one leg was never scored would either
# stop on a missing file that was never expected, or -- worse -- quietly compute
# a two-model ICC while the printout still said three. Reading the roster off the
# files that actually exist means the number of raters is always reported as the
# number of raters there were.
#
# The manifest is consulted first because it carries provenance (which notebook
# wrote the file, when, and its checksum). The folder listing is added as a
# belt-and-braces second pass, so a file copied in by hand is still found.
PREFIX, SUFFIX = "llm_scores_", ".csv"
_man = _load_manifest()
_from_manifest = [fn for fn in _man["files"]
                  if fn.startswith(PREFIX) and fn.endswith(SUFFIX)]
_on_disk = [fn for fn in os.listdir(WORKDIR)
            if fn.startswith(PREFIX) and fn.endswith(SUFFIX)]
SCORE_FILES = sorted(set(_from_manifest) | set(_on_disk))

# Zero files is the one genuinely fatal case, and need() is what says so: it
# stops here with a sentence naming notebook 06 and lists what IS in the folder,
# rather than letting an empty roster surface as a confusing error further down.
if not SCORE_FILES:
    need(f"{PREFIX}<model>{SUFFIX}", produced_by="06")

# The model key is whatever sits between the prefix and the extension:
# llm_scores_claude.csv -> "claude". These keys name every per-model column from
# here to the export, so the ICC, the shared-rater columns and the long file all
# stay traceable to the leg that produced them.
MODELS = [fn[len(PREFIX):-len(SUFFIX)] for fn in SCORE_FILES]
print(f"Found {len(MODELS)} per-model score file(s):")
PATHS = {m: need(f"{PREFIX}{m}{SUFFIX}", produced_by="06") for m in MODELS}

In [ ]:
# === LOAD (1 of 4, cont.): is this the registered ensemble? ===
# Prereg 6.7 registers an ensemble of AT LEAST THREE models, from at least three
# different developers, at least one of them open-weight. That is a measurement
# requirement, not a procurement note:
#   >= 3 MODELS      Two raters give an ICC but no majority -- the holistic call
#                    has no tie-break, and one odd reading moves the mean by half
#                    its distance. Three is the smallest panel on which a single
#                    odd reading is visible AS an odd reading.
#   >= 3 DEVELOPERS  The independence that matters is at the lab level. Two
#                    snapshots from one developer share pretraining data,
#                    tokenizer and post-training, so they share blind spots;
#                    averaging them is closer to sampling one rater twice than to
#                    adding a rater.
#   >= 1 OPEN-WEIGHT Reproducible from published weights by anyone holding the
#                    OSF bundle, and the architecturally most different rater in
#                    the panel.
#
# A short roster does not stop this notebook. It changes what the numbers below
# mean, so it is printed loudly and it must be reported that way in the paper.
if len(MODELS) < 3:
    print("\n  *** THE REGISTERED ENSEMBLE IS INCOMPLETE ***")
    print(f"  Prereg 6.7 registers at least three models from three developers,")
    print(f"  one of them open-weight. This run has {len(MODELS)}: {', '.join(MODELS)}.")
    print("  Everything below still runs. Every number it prints is a number")
    print(f"  about {len(MODELS)} rater(s) and has to be reported as one.")
    if len(MODELS) == 2:
        print("  With two raters the holistic majority has no tie-break: a 1-1")
        print("  split resolves to 'both' where that is one of the two, and")
        print("  alphabetically otherwise. Deterministic, but not a majority.")
    if len(MODELS) == 1:
        print("  With one rater there is NOTHING TO AGREE WITH. Clause B cannot")
        print("  be computed at all, and every facet below reports 'not run'.")
else:
    print(f"\n  Roster of {len(MODELS)}: {', '.join(MODELS)} -- meets the prereg 6.7")
    print("  minimum of three models. Check in the notebook 06 log that they come")
    print("  from three different developers and include one open-weight leg;")
    print("  the file names alone cannot establish that.")

In [ ]:
# === LOAD (2 of 4): what mode was each leg scored in? ===
# RUN_MODE in the config cell describes THIS notebook. It says nothing about the
# files this notebook reads, and done() stamps the two outputs with it either
# way. So a run assembled from smoke legs -- 20 participants apiece -- would be
# recorded in the manifest as "full" unless the legs are asked directly.
#
# They are asked here. Notebook 06 recorded each file's mode in the manifest when
# it wrote it, so the mode is read rather than inferred from row counts. Nothing
# stops on a mismatch: running the whole chain in smoke mode is a normal thing to
# do, and so is re-scoring one leg. What is not acceptable is a mismatch nobody
# saw, because the ICC below is computed listwise -- on the responses EVERY model
# scored -- so one short leg silently sets the sample size for all nine facets.
_files = _load_manifest()["files"]
LEG_MODES = {m: str(_files.get(f"{PREFIX}{m}{SUFFIX}", {}).get("run_mode",
                                                              "unrecorded"))
             for m in MODELS}
LEG_MODES_SEEN = sorted(set(LEG_MODES.values()))
print("Run mode each leg was scored under (as recorded by notebook 06):")
for m in MODELS:
    print(f"  {m:14s} {LEG_MODES[m]}")
if len(LEG_MODES_SEEN) > 1:
    print("\n  *** THE LEGS WERE NOT ALL SCORED IN THE SAME MODE ***")
    print(f"  Modes present: {', '.join(LEG_MODES_SEEN)}. The agreement numbers")
    print("  below are computed on the responses EVERY model scored, so the")
    print("  shortest leg fixes the sample for all nine facets. Re-score the")
    print("  short leg in notebook 06 before reporting anything from this run.")
elif LEG_MODES_SEEN and LEG_MODES_SEEN[0] != RUN_MODE:
    print(f"\n  NOTE: the legs were scored in '{LEG_MODES_SEEN[0]}' mode and this")
    print(f"  notebook is set to RUN_MODE='{RUN_MODE}'. The outputs carry the")
    print("  mode of the SCORES, which is printed in the closing summary.")

In [ ]:
# === LOAD (3 of 4): read each model's file, in the schema notebook 06 writes ===
# THE COLUMN NAMES ALREADY CARRY THE MODEL KEY. Notebook 06 (its cell "fold the
# archive into one row per response x item") writes desert__claude,
# holistic__claude, desert_split__claude, desert_split_noanchor__claude -- never
# a bare "desert" -- so that when the files are pooled here no column can be
# mistaken for another rater's. This cell matches "<stem>__<model>" and renames
# nothing. Matching the bare stems instead would find NO column in any file, and
# an unmatched leg is not an error further down: it is nine columns of NaN, an
# ensemble mean of NaN, and a clause B that quietly never runs while the closing
# summary still reports three raters. That is why an empty match is shouted about
# here rather than noted.
#
# Everything else in the per-model file -- the text, the word count, the vignette
# -- already lives in openends_long.csv, and copying it once per model would put
# three identical text columns in the export.
KEY = ["response_id", "item_type"]
SPLIT_PAT = re.compile(r"^(desert|revenge)_(split|solo)(_noanchor)?$")

def _score_stem(col, m):
    """The stem of a score column belonging to model m, or None."""
    tail = f"__{m}"
    if not str(col).endswith(tail):
        return None
    stem = str(col)[:-len(tail)]
    ok = stem in FACETS or stem == "holistic" or bool(SPLIT_PAT.match(stem))
    return stem if ok else None

raw, UNUSABLE = {}, {}
for m in MODELS:
    fr = pd.read_csv(PATHS[m])
    missing_key = [k for k in KEY if k not in fr.columns]
    keep = [c for c in fr.columns if _score_stem(c, m)]
    if missing_key:
        UNUSABLE[m] = (f"no {', '.join(missing_key)} column, so its rows cannot "
                       "be matched to any other leg")
    elif not keep:
        UNUSABLE[m] = (f"no column called <facet>__{m}. Notebook 06 writes "
                       f"desert__{m}; this file starts: "
                       + ", ".join(map(str, fr.columns[:6])))
    if m in UNUSABLE:
        print(f"  [{m}] UNUSABLE -- {UNUSABLE[m]}")
        continue
    absent = [f for f in FACETS if f"{f}__{m}" not in fr.columns]
    if absent:
        print(f"  [{m}] no column for: {', '.join(absent)} -- unmeasured (NaN)")
    raw[m] = fr[KEY + keep].copy()
    # Facet scores have to be numbers. One stray "n/a" in a column makes pandas
    # read the whole column as text, and a text column would reach the ICC as an
    # object and stop the notebook there. Anything that is not a number becomes
    # NaN, which is this chain's word for UNSCORED, and what was lost is counted
    # out loud rather than absorbed.
    num_cols = [c for c in keep if _score_stem(c, m) != "holistic"]
    _had = int(raw[m][num_cols].notna().sum().sum())
    raw[m][num_cols] = raw[m][num_cols].apply(pd.to_numeric, errors="coerce")
    _lost = _had - int(raw[m][num_cols].notna().sum().sum())
    if _lost:
        print(f"  [{m}] {_lost} score cell(s) were not numbers -- now unscored")
    print(f"  [{m}] {len(raw[m]):,} rows, {len(keep)} score column(s)")

In [ ]:
# === LOAD (3 of 4, cont.): the roster this notebook can actually use ===
# A leg this notebook cannot read is dropped from the roster and named, so every
# number below is a number about the raters that were actually used. If NOTHING
# is readable there is nothing to compute, and that stops the notebook with the
# same kind of sentence need() would have given for a missing file.
MODELS = [m for m in MODELS if m in raw]
if not MODELS:
    raise MissingInput(
        "\n\n  NO USABLE PER-MODEL SCORE FILE\n"
        "  The llm_scores_<model>.csv files are present but none of them carries\n"
        "  the columns notebook 06 writes (desert__<model>, holistic__<model>).\n"
        "  Re-run notebook 06; do not edit the column names by hand.\n"
        "  Reasons, one per file:\n    "
        + "\n    ".join(f"{m}: {why}" for m, why in UNUSABLE.items()) + "\n")
if UNUSABLE:
    print(f"\n  Roster actually used: {', '.join(MODELS)} "
          f"({len(MODELS)} of {len(MODELS) + len(UNUSABLE)} files).")

In [ ]:
# === LOAD (4 of 4): put the legs side by side, one row per response x item ===
# ALIGNMENT IS BY KEY, NEVER BY POSITION. Frames are joined on
# response_id x item_type, the composite key. response_id alone would be wrong:
# one participant wrote up to three texts and carries up to three different
# desert scores. A positional concatenation would look identical right up until
# one model's file lost a row, at which point every score below that row would
# attach to the wrong response and nothing would complain.
#
# THE ONE-ROW-PER-RESPONSE CONTRACT IS ENFORCED HERE, NOT ASSUMED. Notebook 06
# builds its table from a dictionary keyed on (response_id, item_type) and
# asserts its own row count, but a file can be edited, concatenated or appended
# to after 06 wrote it, and this notebook is the one that would launder the
# damage: a single duplicated key in one leg fans out into duplicated rows in
# ensemble_scores.csv, which notebooks 10 and 11 average with pivot_table, giving
# that participant double weight in every gate correlation and every E-number.
# The assert below is what stops that, and it is an assert rather than a warning
# because a doubled participant is not a degraded result, it is a wrong one.
#
# The row universe is the UNION of the keys across legs, built once, so a
# response only one model returned still gets a row (with NaN for the others).
# NaN means UNSCORED and must never become 0 -- a zero reads downstream as "this
# facet is absent from the passage" instead of "no rater ever saw this passage",
# and the two are not the same claim.
scored = (pd.concat([raw[m][KEY] for m in MODELS], ignore_index=True)
            .drop_duplicates().reset_index(drop=True))
for m in MODELS:
    dup = raw[m].duplicated(KEY, keep=False)
    assert not dup.any(), (
        f"{PREFIX}{m}{SUFFIX} has {int(dup.sum())} rows sharing a "
        f"response_id x item_type key, e.g.\n"
        f"{raw[m].loc[dup, KEY].head(4).to_string(index=False)}\n"
        "  One row per response per item is the contract every file in this "
        "chain is keyed on. Re-run notebook 06 rather than de-duplicating here: "
        "a duplicate means its archive holds two records of the same pass.")
    n_before = len(scored)
    scored = scored.merge(raw[m], on=KEY, how="left")
    assert len(scored) == n_before, (
        f"Joining {m} changed the row count from {n_before} to {len(scored)}. "
        "A left join onto a fixed set of keys cannot do that unless the right "
        "side repeats a key -- which the check above is meant to have caught.")

# How many responses the WHOLE panel returned. This is the sample every ICC below
# is estimated on (listwise: a response counts only where every model answered),
# so it is worth seeing before any agreement number. Zero would mean the legs
# describe disjoint sets of responses -- different id formats, or two different
# corpora -- which produces no error anywhere and no ICC either.
_seen = [raw[m][KEY].drop_duplicates().assign(**{f"_in_{m}": 1}) for m in MODELS]
_panel = _seen[0]
for _s in _seen[1:]:
    _panel = _panel.merge(_s, on=KEY, how="inner")
N_ALL_LEGS = len(_panel)
print(f"\nMerged frame: {scored.shape[0]:,} rows (response x item) x "
      f"{scored.shape[1]} columns.")
print(f"  scored by at least one model: {len(scored):,}")
print(f"  scored by all {len(MODELS)} model(s):    {N_ALL_LEGS:,}"
      + ("   <-- the sample every ICC below rests on" if len(MODELS) > 1 else ""))
if len(MODELS) > 1 and N_ALL_LEGS == 0:
    print("\n  *** NO RESPONSE WAS SCORED BY EVERY MODEL ***")
    print("  The legs do not overlap at all, so no ICC can be computed on any")
    print("  facet. Check that the files describe the same corpus and that")
    print("  response_id is written the same way in each.")

In [ ]:
# === LOAD (4 of 4, cont.): how much of the corpus each model actually scored ===
# WHAT THIS PRINTOUT MEANS, and why it comes before any agreement number.
# A model that failed to return a usable answer on some responses contributes
# NaN there, and those rows drop out of the ICC listwise -- "listwise" meaning
# the whole response is set aside if any one model is missing on it. So the
# coverage table is the first thing to read when an ICC looks unstable: a leg at
# 60% coverage is not a leg that disagreed, it is a leg that was largely absent,
# and the two call for completely different responses.
#
# What to look for: a coverage figure well below the others usually means that
# leg hit rate limits, refusals on the violent content, or truncated replies --
# all of which are visible in the notebook 06 failure log, and none of which are
# visible here.
cov_rows = []
for m in MODELS:
    facet_cols = [f"{f}__{m}" for f in FACETS if f"{f}__{m}" in scored.columns]
    any_facet = (scored[facet_cols].notna().any(axis=1) if facet_cols
                 else pd.Series(False, index=scored.index))
    hol_col = f"holistic__{m}"
    cov_rows.append({
        "model": m,
        "rows_in_file": len(raw[m]),
        "rows_with_any_facet": int(any_facet.sum()),
        "coverage": float(any_facet.mean()) if len(scored) else np.nan,
        "rows_with_holistic": (int(scored[hol_col].notna().sum())
                               if hol_col in scored.columns else 0),
    })
coverage_df = pd.DataFrame(cov_rows)
print("Per-model coverage of the merged corpus:")
# na_rep is what renders an uncomputable cell as "--". pandas passes nulls
# through na_rep BEFORE float_format ever sees them, so a null test inside the
# formatter is dead code and the table prints "NaN" instead.
print(coverage_df.to_string(index=False, na_rep="--",
                            float_format=lambda x: f"{x:.1%}"))
if len(scored) and coverage_df["coverage"].min() < 0.90:
    print("\n  NOTE: at least one leg scored under 90% of the corpus. Read the")
    print("  ICC below against this table, and check the notebook 06 failure log.")

## Do

In [ ]:
# === DO (1 of 9): the facet score of record -- the MEAN across models ===
# Prereg 6.7: the facet score is the mean across models, the holistic call is the
# majority. This cell does the first of those.
#
# skipna=True is the mechanism that makes the unscored-row policy work end to
# end: a response two models scored and one failed on gets the mean of the two
# that answered, not a mean dragged toward zero by a phantom third rater. The
# cost is that different rows can rest on different numbers of raters, which is
# why n_models_scoring is carried beside every mean rather than left implicit.
#
# THE ENSEMBLE MEAN IS COMPUTED HERE AND TRUSTED NOWHERE YET.
#   Prereg 6.7: "Agreement among the models is reliability, not validity." A mean
#   across three models that share a blind spot is not more accurate than any one
#   of them -- it is the same error with a tighter confidence interval, which is
#   worse, because it reads as consensus. Every rater is kept individually
#   addressable all the way to the export for the same reason: collapsing to the
#   mean here would throw away the only evidence that could ever reveal a shared
#   reading.
for f in FACETS:
    cols = [f"{f}__{m}" for m in MODELS if f"{f}__{m}" in scored.columns]
    if cols:
        scored[f"{f}_ens_mean"] = scored[cols].mean(axis=1, skipna=True)
        scored[f"{f}_n_models"] = scored[cols].notna().sum(axis=1)
    else:
        scored[f"{f}_ens_mean"] = np.nan
        scored[f"{f}_n_models"] = 0

# PREVALENCE IS A SHARE OF THE RESPONSES THAT WERE SCORED, never of all rows.
# An unscored response has no ensemble mean, and NaN > 0 is False, so counting it
# in the denominator would report it as "this facet is absent here" -- turning an
# absence of measurement into a measured base rate. On a run where a leg failed
# outright that reads as a confident 0.0% for every facet. The denominator is
# printed beside the percentage so the two can never be confused again.
prev, prev_clear, n_scored = {}, {}, {}
for f in FACETS:
    _seen = scored[f"{f}_ens_mean"].notna()
    n_scored[f] = int(_seen.sum())
    prev[f] = float((scored.loc[_seen, f"{f}_ens_mean"] > 0).mean()) if _seen.any() else np.nan
    # Clear presence on the same denominator: the ensemble mean above PRESENT_CLEAR.
    prev_clear[f] = (float((scored.loc[_seen, f"{f}_ens_mean"] > PRESENT_CLEAR).mean())
                     if _seen.any() else np.nan)
print("Ensemble mean computed for nine facets. Share of the SCORED responses")
print("that score above zero (prevalence, for context on every ICC below -- a")
print("facet almost nobody expresses has almost no variance to agree about):")
for f in FACETS:
    _p = "--" if pd.isna(prev[f]) else f"{prev[f]:6.1%}"
    _c = "--" if pd.isna(prev_clear[f]) else f"{prev_clear[f]:6.1%}"
    print(f"  {f:16s} trace {_p}  clear {_c}  of {n_scored[f]:,} scored"
          + ("" if n_scored[f] == len(scored) else f" (of {len(scored):,} rows)"))

In [ ]:
# === DO (2 of 9): the holistic call of record -- the MAJORITY across models ===
# The holistic judgment is nominal (principled | vengeful | both | neither), so
# majority rather than mean.
#
# TIE RULE, stated once so it is not a hidden choice: with three raters a tie
# means 1/1/1 (or 2/2 with four raters), and 'both' wins if it is among the tied
# levels. That is the conservative reading -- raters split between 'principled'
# and 'vengeful' HAVE described a passage that shows both -- and it is
# deterministic rather than order-dependent. Otherwise the alphabetically first
# tied level is taken, which is arbitrary but stable across re-runs. Values
# outside the four levels are dropped first, and a response no model called
# returns None.
#
# HAZARD: this column is descriptive coverage and a Krippendorff target only. It
# comes from the same pass and the same rater as the facet vector, so it can
# never corroborate facet-level co-occurrence for E4 -- that would be one read
# counted twice (design section 9 rule 6).
def _majority(vals):
    vals = [v for v in vals if v in HOLISTIC_LEVELS]
    if not vals:
        return None
    c = Counter(vals)
    top = c.most_common()
    best = top[0][1]
    tied = [lvl for lvl, n in top if n == best]
    return "both" if "both" in tied else sorted(tied)[0]

HOL_MODELS = [m for m in MODELS if f"holistic__{m}" in scored.columns]
if HOL_MODELS:
    scored["holistic_majority"] = [
        _majority([scored.loc[i, f"holistic__{m}"] for m in HOL_MODELS])
        for i in scored.index
    ]
    got = scored["holistic_majority"].notna().sum()
    print(f"Holistic majority from {len(HOL_MODELS)} model(s): {got} of {len(scored)} responses called.")
    # The responses no model called are labelled rather than left as NaN: a
    # blank in a count table cannot be told apart from a category of zero.
    print(scored["holistic_majority"].fillna("(not called by any model)")
          .value_counts().to_string())
else:
    # Not a failure and not a stop: the facets are the measure, the holistic call
    # is descriptive coverage. It is recorded as absent rather than as empty.
    scored["holistic_majority"] = None
    print("No model file carried a holistic column. The holistic call is NOT RUN;")
    print("Krippendorff's alpha below reports 'not run' rather than a blank.")

In [ ]:
# === DO (3 of 9): the separate-call (split) scores for the E4 pair, if present ===
# WHY THESE EXIST. Scoring all nine facets in ONE call means desert and revenge
# share the rater's overall impression of the passage, the same window of text,
# and a rubric that explicitly says real explanations mix motives. E4's whole
# hypothesis IS the desert x revenge association, so producing both numbers from
# one read lets a within-rater halo manufacture the effect. Notebook 06 therefore
# scores desert and revenge again in separate, independent calls -- and once more
# with the co-occurrence sentence stripped out, because a co-occurrence that
# depends on the rubric's own co-occurrence instruction is circular (design
# section 4). If the no-anchor correlation drops below the split correlation, the
# rubric's teaching was contributing, and that contribution is a reportable
# sensitivity rather than a nuisance.
#
# They are carried under DISTINCT names so a joint score and a separate-call
# score can never be selected for one another downstream. The joint columns stay
# untouched: they are the preregistered estimate and must remain reportable on
# their own.
#
# If notebook 06 did not write them, this prints one line and carries on. The
# halo diagnostic that consumes them lives in the E4 notebook, not here.
EXTRA_ELICITATIONS = {}          # long-file row label -> per-model column stem
for f in ("desert", "revenge"):
    for label, stems in ((f"{f}_solo", [f"{f}_split", f"{f}_solo"]),
                         (f"{f}_solo_noanchor",
                          [f"{f}_split_noanchor", f"{f}_solo_noanchor"])):
        for stem in stems:
            cols = [f"{stem}__{m}" for m in MODELS if f"{stem}__{m}" in scored.columns]
            if cols:
                # Same mean-across-models rule as the joint call, same skipna
                # policy, so the two elicitations of one facet stay comparable.
                scored[f"{label}_ens_mean"] = scored[cols].mean(axis=1, skipna=True)
                scored[f"{label}_n_models"] = scored[cols].notna().sum(axis=1)
                EXTRA_ELICITATIONS[label] = stem
                break

# The labels become facet values in the long export, so they must not collide
# with the nine facet names -- a collision would put two different measurements
# under one facet label and the export would carry the same key twice.
assert not (set(EXTRA_ELICITATIONS) & set(FACETS)), (
    "A separate-call label collides with a facet name: "
    f"{sorted(set(EXTRA_ELICITATIONS) & set(FACETS))}")

if EXTRA_ELICITATIONS:
    print("Separate-call columns found and averaged:")
    for label, stem in EXTRA_ELICITATIONS.items():
        print(f"  {label:22s} from {stem}__<model>")
else:
    print("No separate-call (split) columns in the per-model files. The joint")
    print("nine-facet scores are exported on their own; the E4 halo diagnostic")
    print("needs the split columns and will report them as not available.")

In [ ]:
# === DO (4 of 9): when an ICC is arithmetic rather than a measurement ===
# An intraclass correlation is a RATIO: the share of the variation in the scores
# that is real differences between responses rather than differences between the
# raters reading them. Every ratio has a denominator, and on some facets that
# denominator is zero or all but zero. The arithmetic still returns a number.
# Three cases, and this corpus can produce all three:
#
#   EVERY RATER GAVE THE SAME VALUE EVERYWHERE. Nothing varies, so there is
#   nothing to apportion, and the library divides zero by zero. On a facet scored
#   flat 0.0 that returns NaN; on a facet scored flat 0.30 what actually gets
#   divided is the rounding dust left over from the mean, and the answer comes
#   back 1.000 -- perfect agreement, computed on floating-point noise. Trusting
#   the library to return NaN here is trusting an accident.
#
#   THE RESPONSES DO NOT DIFFER, ONLY THE RATERS DO. The same empty denominator
#   from the other side: no between-response variation for the ICC to be a share
#   of.
#
#   ALMOST NOBODY EXPRESSES THE FACET. Five of 900 responses above zero, the
#   panel agreeing on the other 895 zeros, gives a genuine ICC of 1.000 resting
#   on five responses. That one is not undefined. It is unearned, and it is
#   caught by the positive-case floor in the clause B cell rather than here.
#
# The first two are named by this function so that they can be reported as "not
# run" with the reason, instead of arriving as a number that reads as agreement.
def _no_variance_reason(d):
    """Name the degeneracy in a responses x raters frame, or return ''."""
    vals = d.to_numpy(dtype=float)
    if float(np.ptp(vals)) < 1e-12:
        return (f"every model returned {vals.flat[0]:.2f} on every response "
                "-- there is no variation to apportion")
    if float(d.mean(axis=1).var(ddof=1)) < 1e-12:
        return ("the responses do not differ from one another, only the models "
                "do -- an ICC has no between-response variation to be a share of")
    return ""

print("Degeneracy check defined: it names the cases where an ICC is undefined,")
print("rather than leaving them to come back as 1.000 or as NaN by accident.")

In [ ]:
# === DO (5 of 9): the two intraclass correlations ===
def icc_2k_21(frame):
    """Cross-model ICC for one facet, as a dict of numbers and reasons.

    Two-way random effects, absolute agreement, models as raters and responses
    as targets. Both forms are reported because they answer different questions
    (prereg 6.7, design section 7):
      ICC(2,k)  reliability of the k-model MEAN, which is the score actually
                used downstream -- the honest reliability of the measure of
                record
      ICC(2,1)  reliability of a SINGLE model, which is the honest reliability
                of the underlying rater and does not improve just because more
                raters were bought
    ICC(2,k) is always the larger of the two; quoting it alone would let a panel
    of mediocre raters look like a good instrument.

    Absolute agreement rather than consistency: a model that rank-orders
    responses perfectly but sits half a point high is not interchangeable with
    the others, and the mean across models would inherit the offset.
    """
    # Listwise across models: a response counts toward the ICC only where every
    # model returned a value. This is the one place the unscored rows narrow the
    # sample rather than being skipped pairwise, and it is deliberate -- an ICC
    # is a statement about a fixed panel of raters, so it has to be estimated on
    # responses the whole panel actually rated. If one leg's coverage was poor,
    # this frame gets small, and that shows up as an unstable ICC rather than as
    # a silently different quantity.
    d = frame.dropna()
    out = {"icc2k": np.nan, "icc21": np.nan, "n_used": len(d), "n_pos": 0,
           "undefined_reason": "", "warned": ""}
    if d.shape[1] < 2 or len(d) < 5:
        return out
    # How many of those responses express the facet at all. Carried out of here
    # because the clause B verdict needs it: agreement about zeros is agreement
    # about nothing.
    out["n_pos"] = int((d.to_numpy(dtype=float) > 0).any(axis=1).sum())
    out["undefined_reason"] = _no_variance_reason(d)
    if out["undefined_reason"]:
        return out
    # pingouin wants long form: one row per response x model.
    long = d.reset_index(drop=True).reset_index().melt(
        id_vars="index", var_name="rater", value_name="score")
    # The divide-by-zero and invalid-value warnings pingouin raises on a
    # degenerate facet are evidence, so they are CAUGHT AND CARRIED rather than
    # silenced. record=True collects them instead of printing them once and
    # never again; "always" is needed because Python's default is to show a
    # repeated warning once, which would leave the second bad facet looking
    # clean.
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            icc = pg.intraclass_corr(data=long, targets="index",
                                     raters="rater", ratings="score")
            i2k = icc[icc["Type"] == "ICC2k"]["ICC"]
            i21 = icc[icc["Type"] == "ICC2"]["ICC"]
            out["icc2k"] = float(i2k.iloc[0]) if len(i2k) else np.nan
            out["icc21"] = float(i21.iloc[0]) if len(i21) else np.nan
        except Exception as e:
            out["undefined_reason"] = f"pingouin could not fit it ({type(e).__name__})"
    out["warned"] = "; ".join(sorted({str(w.message) for w in caught
                                      if issubclass(w.category, RuntimeWarning)}))
    return out

print("icc_2k_21 defined: ICC(2,k) for the ensemble mean, ICC(2,1) for one model.")

In [ ]:
# === DO (6 of 9): clause B, per facet ===
# Prereg 6.7 gate clause B: "the models agree at an intraclass correlation of at
# least .60". Computed here for all nine facets, including the descriptive-only
# ones (proportionality, censure, moral_balance), so the demotion of those facets
# stays auditable rather than assumed.
#
# WHERE A CLAUSE DID NOT RUN IT SAYS SO. clauseB is a word, not a blank: "pass",
# "fail", or "not run" with the reason beside it. A NaN in a gate column reads as
# a failure to anyone skimming the table, and a facet that could not be evaluated
# is not a facet that failed.
#
# THE ORDER OF THE TESTS IS THE POINT. Every floor is checked BEFORE the .60
# comparison, because each one describes a situation in which the ICC is a number
# without being a measurement -- and in most of them the number the arithmetic
# returns is 1.000, which is to say "pass". A facet five people wrote about would
# otherwise clear the registered reliability clause on the strength of the 895
# zeros the models agreed about, and would then be carried into the gate notebook
# and the OSF bundle as a reliable measure.
rel_rows = []
for f in FACETS:
    cols = [f"{f}__{m}" for m in MODELS if f"{f}__{m}" in scored.columns]
    r = icc_2k_21(scored[cols]) if len(cols) >= 2 else None
    if r is None:
        # No n_complete either: a sample size printed beside a verdict that was
        # never computed reads as evidence standing behind it.
        i2k = i21 = n_used = n_pos = np.nan
        warned, status = "", "not run"
        if len(cols) == 1:
            note = "only one model scored this facet, so there is nothing to agree with"
        else:
            note = "no model file carried a column for this facet"
    else:
        i2k, i21 = r["icc2k"], r["icc21"]
        n_used, n_pos, warned = r["n_used"], r["n_pos"], r["warned"]
        if n_used < 5:
            status, note = "not run", f"only {n_used} responses scored by every model"
        elif r["undefined_reason"]:
            status, note = "not run", "ICC undefined: " + r["undefined_reason"]
        elif n_pos < MIN_POSITIVE_CASES:
            status = "not run"
            note = (f"only {n_pos} of {n_used} responses express this facet at "
                    f"all, below the {MIN_POSITIVE_CASES}-case floor -- an "
                    "agreement about zeros")
        elif pd.isna(i2k):
            status, note = "not run", "the ICC came back undefined"
        elif i2k >= ICC_THRESHOLD:
            status, note = "pass", f"ICC(2,k) at or above the registered {ICC_THRESHOLD:.2f}"
        else:
            status, note = "fail", f"ICC(2,k) below the registered {ICC_THRESHOLD:.2f}"
    rel_rows.append({"facet": f, "n_models": len(cols), "n_complete": n_used,
                     "n_positive": n_pos, "icc2k": i2k, "icc21": i21,
                     "clauseB": status, "clauseB_note": note,
                     "clauseB_basis": "point estimate of ICC(2,k)",
                     "icc_threshold": ICC_THRESHOLD,
                     "min_positive_cases": MIN_POSITIVE_CASES,
                     "prevalence_nonzero": prev[f], "prevalence_clear": prev_clear[f],
                     "present_clear_threshold": PRESENT_CLEAR,
                     "n_scored": n_scored[f],
                     "icc_warning": warned, "models": "|".join(MODELS),
                     "leg_run_modes": "|".join(LEG_MODES[m] for m in MODELS)})
print(f"Clause B computed for {len(rel_rows)} facets across {len(MODELS)} model(s).")

In [ ]:
# === DO (6 of 9, cont.): the clause B table, and what its columns promise ===
inter_model_icc = pd.DataFrame(rel_rows)
# Counts are whole numbers, and "no count" has to survive as empty rather than as
# 0. Int64 (capital I) is the pandas integer type that admits a missing value; a
# plain int column cannot hold one and would turn every not-run row into a zero
# sample size, which reads as a measurement of nothing rather than as no
# measurement.
for _c in ("n_complete", "n_positive"):
    inter_model_icc[_c] = inter_model_icc[_c].astype("Int64")

# A PASS THAT RESTS ALMOST ENTIRELY ON ZEROS IS STILL A PASS. The .60 threshold
# is the registration's and this notebook does not move it, so a facet above the
# positive-case floor keeps its verdict. It is FLAGGED rather than overruled: a
# facet 40 of 2,844 people wrote about can post an ICC of 1.000 because the panel
# agreed on the 2,804 zeros, and a reader looking at the ICC column alone has no
# way to see that. 5% is the same floor design section 6 puts on the word list
# before its agreement counts as corroboration, applied here for the same reason.
# A facet whose counts are empty gets a share of NaN, and NaN < 0.05 is False, so
# a clause that never ran is never also flagged as a thin pass.
_share = (inter_model_icc["n_positive"] / inter_model_icc["n_complete"]).astype(float)
inter_model_icc["low_prevalence"] = (inter_model_icc["clauseB"].eq("pass")
                                     & (_share < 0.05))
inter_model_icc.loc[inter_model_icc["low_prevalence"], "clauseB_note"] += (
    "; but under 5% of the responses express it, so most of the agreement is "
    "agreement about zeros")

# One row per facet is the key of this file, and the gate notebook selects on it
# with inter_model_icc[inter_model_icc["facet"] == f]. Two rows for one facet
# would silently hand it the first one.
assert not inter_model_icc.duplicated(["facet"]).any(), (
    "inter_model_icc has more than one row for a facet: "
    f"{inter_model_icc.loc[inter_model_icc.duplicated(['facet']), 'facet'].tolist()}. "
    "FACETS in the setup cell must list each facet exactly once.")

# THERE IS DELIBERATELY NO item_type COLUMN. These numbers are pooled across the
# three items: one ICC per facet, computed on backward, ranking and forward
# together. Adding the column would claim a per-item estimate this cell never
# computed. The gate notebook reuses the one facet row at each item_type on
# purpose, and says so where it does it.
#
# WHAT clauseB IN THIS FILE MEANS, precisely. It is the verdict on the POINT
# ESTIMATE of ICC(2,k), which is the quantity prereg 6.7 names. The gate notebook
# computes its own clause B on the LOWER BOUND of a bootstrap confidence interval
# -- a stricter test, registered as a deviation in design section 7 -- and that
# one is the decision of record. Both travel under the name clauseB in their own
# files, so the basis of each is written into the row (clauseB_basis) rather than
# left to the reader to remember. The union of the clauses is applied once, in
# the gate notebook, on this file: combining them in two places is how two copies
# of one decision drift apart.
print(f"Clause B table: {len(inter_model_icc)} rows x "
      f"{inter_model_icc.shape[1]} columns, one row per facet, pooled over items.")

In [ ]:
# === DO (7 of 9): Krippendorff's alpha on the holistic call ===
# Prereg 6.7 is what puts this coefficient in the pipeline, in as many words:
# "an intraclass correlation for the 0-to-1 scores and Krippendorff's alpha for
# the categorical judgment, which plays the part that inter-rater reliability
# plays with human coders."
#
# The holistic judgment is nominal (principled | vengeful | both | neither), so
# an ICC is meaningless on it. Krippendorff's alpha -- a measure of how much
# independent raters agree, beyond what they would hit by chance -- is the right
# coefficient here: it handles labels rather than numbers, more than two raters,
# and missing values, which matters because a model that failed on a response
# contributes nothing for it.
#
# The four levels are mapped to integer codes only to get a numeric matrix; the
# level_of_measurement="nominal" argument is what keeps those codes from being
# read as an ordering (there is no sense in which 'both' sits between 'vengeful'
# and 'neither'). The matrix is raters x units, the layout the library expects --
# transposing it would compute agreement across responses instead of across
# models and return a number with no interpretation.
#
# Wrapped in try/except because alpha is undefined when every model returned the
# same level for every response. "not run" is the honest answer there.
_code = {lvl: i for i, lvl in enumerate(HOLISTIC_LEVELS)}
holistic_alpha, alpha_status = np.nan, "not run"
alpha_note = "no holistic column in the per-model files"
if len(HOL_MODELS) >= 2:
    hol_matrix = np.array(
        [[_code.get(v, np.nan) for v in scored[f"holistic__{m}"]] for m in HOL_MODELS],
        dtype=float)
    try:
        holistic_alpha = krippendorff.alpha(reliability_data=hol_matrix,
                                            level_of_measurement="nominal")
        alpha_status, alpha_note = "computed", ""
    except Exception as e:
        alpha_note = f"alpha undefined ({type(e).__name__})"
elif len(HOL_MODELS) == 1:
    alpha_note = "only one model made the holistic call -- nothing to agree with"

# Carried as a column on every row of the gate file, with the same value
# throughout: it is ONE number for the whole holistic call, not a per-facet
# quantity. It travels in this file because that is where the gate notebook looks
# for it, and because a reliability number reported nowhere gets reported never.
inter_model_icc["kripp_alpha_holistic"] = holistic_alpha
inter_model_icc["kripp_alpha_status"] = alpha_status
inter_model_icc["kripp_alpha_note"] = alpha_note
print("Holistic Krippendorff alpha (nominal, across models): "
      + (f"{holistic_alpha:.3f}" if pd.notna(holistic_alpha)
         else f"not run -- {alpha_note}"))

In [ ]:
# === DO (8 of 9): read the clause B table ===
print("GATE CLAUSE B -- do the models agree with each other?")
print(f"(prereg 6.7: pass at ICC(2,k) >= {ICC_THRESHOLD:.2f}, on at least "
      f"{MIN_POSITIVE_CASES} responses that express the facet.")
print(f" Raters: {len(MODELS)} -- {', '.join(MODELS)})\n")
_show = inter_model_icc[["facet", "n_models", "n_complete", "n_positive", "icc2k",
                         "icc21", "clauseB", "prevalence_nonzero",
                         "prevalence_clear"]].copy()
# Whole-number columns are turned into text first: na_rep governs how pandas
# renders a missing FLOAT, and a missing Int64 would print as <NA> beside it.
# Both must read "--", which is the doctrine for this whole chain -- a quantity
# that could not be computed shows as "--", never as a blank and never as NaN.
for _c in ("n_complete", "n_positive"):
    _show[_c] = _show[_c].map(lambda v: "--" if pd.isna(v) else f"{int(v):,}")
print(_show.to_string(index=False, na_rep="--",
                      float_format=lambda x: f"{x:.3f}"))

# The reasons are printed under the table rather than in it: a sentence long
# enough to be worth reading is too long to sit in a column, and the truncated
# version pandas would show is the part that matters least. Every row that is not
# a plain unqualified pass gets its sentence here.
_why = inter_model_icc[inter_model_icc["clauseB"].ne("pass")
                       | inter_model_icc["low_prevalence"]]
if len(_why):
    print("\nWHY, for every facet whose verdict is not a plain pass:")
    for _, _r in _why.iterrows():
        print(f"  {_r['facet']:16s} [{_r['clauseB']}] {_r['clauseB_note']}")

# WHAT THIS TABLE IS, AND WHAT IT IS NOT. Every number above is agreement of the
# models WITH EACH OTHER. That is consistency, not accuracy. Prereg 6.7:
# "Agreement among the models is reliability, not validity." Three models trained
# on similar text can read "pay for what he did" as pure desert where many
# writers mean payback, and they will then produce three matching numbers, a low
# between-model variance and a high ICC -- confidently wrong, and this table
# cannot tell the difference.
print("\nHOW TO READ IT")
print("  ICC(2,k)  reliability of the mean across models -- the score of record.")
print("  ICC(2,1)  reliability of ONE model. Always the smaller of the two, and")
print("            it does not improve by buying more raters.")
print("  n_positive  how many of the responses the whole panel scored express")
print(f"            the facet at all. Below {MIN_POSITIVE_CASES} the clause is")
print("            NOT RUN: agreement about zeros is agreement about nothing.")
print("  This is agreement of the models WITH EACH OTHER: consistency, not")
print("  accuracy. A high ICC on a facet the whole panel misreads the same way")
print("  is exactly what a shared blind spot looks like from here.")
if not HUMAN_GOLD:
    print("  HUMAN_GOLD is False, so no human-coded comparison runs anywhere in")
    print("  this chain: clause A is NOT RUN, and there is no independent check")
    print("  on shared bias. Clause B on its own licenses nothing (design 4, 7).")
else:
    print("  HUMAN_GOLD is True: clause A is computed in the optional human-coder")
    print("  notebook and joined to this table by the gate notebook.")
print("  Clause C (each facet correlates r >= .30 with its matching survey")
print("  scale) is NOT RUN here -- it is a separate notebook. The gate is the")
print("  union of the clauses, and no clause is the gate on its own.")

In [ ]:
# === DO (8 of 9, cont.): the one-line verdict count, and any library warning ===
# Every ICC above was estimated listwise, on the responses the whole panel
# returned. Where that is a small part of the corpus, every number in the table
# is about that part -- most often because one leg was scored in smoke mode or
# hit rate limits, both of which are visible further up rather than here.
if len(scored) and N_ALL_LEGS < 0.9 * len(scored):
    print(f"  SAMPLE BEHIND EVERY ICC ABOVE: {N_ALL_LEGS:,} of {len(scored):,} "
          f"responses ({N_ALL_LEGS / len(scored):.0%}), because that is how many")
    print("  every model scored. Read the coverage table and the leg modes above.")
low = inter_model_icc[inter_model_icc["clauseB"] == "fail"]["facet"].tolist()
notrun = inter_model_icc[inter_model_icc["clauseB"] == "not run"]["facet"].tolist()
print(f"  clause B pass:    {inter_model_icc['clauseB'].eq('pass').sum()} facets")
print(f"  clause B fail:    {', '.join(low) if low else 'none'}")
print(f"  clause B not run: {', '.join(notrun) if notrun else 'none'}")

# The passes that rest on almost nothing, named. They are still passes; what is
# not acceptable is one that nobody noticed was carried by the zeros.
thin = inter_model_icc[inter_model_icc["low_prevalence"]]["facet"].tolist()
if thin:
    print(f"\n  PASSED, BUT ON ALMOST NO POSITIVE CASES: {', '.join(thin)}")
    print("  Under 5% of the responses these models scored express the facet at")
    print("  all, so the ICC is mostly agreement about zeros. The gate notebook")
    print("  puts a confidence interval around each of these; expect it to be")
    print("  wide, and read the interval rather than the point estimate.")

# Any arithmetic warning the ICC library raised is printed rather than swallowed.
# "Divide by zero" means one of the variances in the formula was exactly zero,
# and it has two very different readings, which is why the warning is shown
# rather than acted on:
#   the models returned IDENTICAL scores, response by response. The error term is
#   zero because the agreement is perfect. The ICC of 1.000 is real, and worth
#   knowing about for a different reason -- three raters that never differ by a
#   hundredth are behaving like one rater.
#   or the facet has almost nothing in it, and the zeros are doing the work. The
#   floors above are what catch that, so check n_positive on the same row.
_warned = inter_model_icc[inter_model_icc["icc_warning"].astype(str).str.len() > 0]
if len(_warned):
    print("\n  THE ICC LIBRARY WARNED WHILE COMPUTING THESE FACETS:")
    for _, _r in _warned.iterrows():
        print(f"    {_r['facet']:16s} [{_r['clauseB']}] n_positive="
              f"{_r['n_positive']}  {_r['icc_warning']}")
    print("  Read each against its n_positive: perfect agreement and no facet to")
    print("  agree about produce the same warning and mean opposite things.")
else:
    print("\n  The ICC library raised no arithmetic warnings on any facet.")

In [ ]:
# === DO (9 of 9): melt the scores into long form ===
# WHY LONG. Facet is a factor in every downstream analysis -- E3 compares facets
# between groups, E6 differences a facet across item_type, E7 adds facets to a
# regression -- and the gate is applied per facet. Long form makes facet a VALUE
# that can be grouped, filtered and joined against the gate table, rather than a
# column name that has to be parsed.
#
# WHAT EACH ROW CARRIES, and why nothing is filtered out here:
#   ensemble_mean      the joint-call score of record for that facet
#   n_models_scoring   how many raters that particular mean rests on
#   holistic           the majority call for the response (repeated per facet)
#   score_<model>      every rater individually, so any later check of shared
#                      error or of the open-weight leg on its own remains
#                      reproducible from this one file
#
# Plus, where notebook 06 produced them, extra ROWS per response carrying the
# separate-call scores under facet='desert_solo' / 'revenge_solo' (and the
# no-anchor variants). They are rows and not columns on purpose: the E4 notebook
# selects its scores by facet, and a score that lives only in a column never
# reaches an analysis frame built that way.
#
# FACETS THAT FAIL A GATE CLAUSE ARE EXPORTED WITH THEIR SCORES INTACT, never
# dropped. A file containing only what passed makes the gate invisible to anyone
# reading the data, and the descriptive-only facets (proportionality, censure,
# moral_balance) still carry prevalence and register results inside the
# exploratory family.
long_rows = []
for _, row in scored.iterrows():
    for f in list(FACETS) + list(EXTRA_ELICITATIONS.keys()):
        stem = EXTRA_ELICITATIONS.get(f, f)     # solo rows read their own stem
        rec = {"response_id": row["response_id"],
               "item_type": row["item_type"],
               "facet": f,
               "ensemble_mean": row.get(f"{f}_ens_mean"),
               "n_models_scoring": row.get(f"{f}_n_models"),
               "holistic": row.get("holistic_majority")}
        for m in MODELS:
            rec[f"score_{m}"] = row.get(f"{stem}__{m}")
        long_rows.append(rec)
ensemble_scores = pd.DataFrame(long_rows)

In [ ]:
# === DO (9 of 9, cont.): the contract this file is written under ===
# ONE ROW PER response_id x item_type x facet, AND NO REPEATS. Every consumer of
# ensemble_scores.csv assumes it. Notebook 99 pivots the file with .pivot(), which
# raises on a repeat; notebooks 10 and 11 pivot with .pivot_table(), which does
# not raise -- it AVERAGES the duplicates, giving that participant double weight
# in every gate correlation and every exploratory test while the printed sample
# size stays the same. That is a wrong number that looks like a right one, which
# is why 10 and 11 each check the file for repeats as they read it. This is the
# writing half of the same contract: the file is checked before it is written, by
# the notebook that wrote it.
#
# The response_id x item_type half is enforced on the way IN, in the LOAD cell
# that joins the legs. What this assertion adds is the facet dimension: a facet
# name repeated in FACETS, or a separate-call label that collides with one, would
# write the same key twice from a perfectly clean set of inputs.
_dups = ensemble_scores.duplicated(["response_id", "item_type", "facet"], keep=False)
assert not _dups.any(), (
    f"{int(_dups.sum())} rows share a response_id x item_type x facet key, e.g.\n"
    f"{ensemble_scores.loc[_dups, ['response_id', 'item_type', 'facet']].head(4).to_string(index=False)}\n"
    "  Either FACETS lists a facet twice, or a separate-call label matches a "
    "facet name, or a per-model file arrived with a duplicated key.")

_per_response = len(FACETS) + len(EXTRA_ELICITATIONS)
print(f"Long frame ready: {len(ensemble_scores):,} rows = {len(scored):,} "
      f"responses x ({len(FACETS)} facets + {len(EXTRA_ELICITATIONS)} "
      "separate-call rows).")
print(f"  unique response x item x facet keys: "
      f"{ensemble_scores[['response_id', 'item_type', 'facet']].drop_duplicates().shape[0]:,}"
      f"  (must equal the row count, and {len(scored):,} x {_per_response})")

## Save and check

In [ ]:
# === SAVE AND CHECK ===
# Two files leave this notebook.
#   ensemble_scores.csv   the facet score of record, long form, every rater kept
#                         individually addressable beside the mean
#   inter_model_icc.csv   gate clause B per facet, plus the holistic alpha
#
# Both are written first and registered with done() second, so the manifest can
# never claim a file that is not on disk.
#
# The note on each file records the mode the SCORES were produced in, not just
# this notebook's RUN_MODE. done() stamps its own run_mode, and an ensemble built
# from smoke legs would otherwise be filed as "full".
_leg_note = "; ".join(f"{m}={LEG_MODES[m]}" for m in MODELS)
ensemble_scores.to_csv("ensemble_scores.csv", index=False)
done("ensemble_scores.csv", rows=len(ensemble_scores),
     note=(f"long: response x item x facet; mean across {len(MODELS)} model(s); "
           f"leg modes: {_leg_note}"))

inter_model_icc.to_csv("inter_model_icc.csv", index=False)
done("inter_model_icc.csv", rows=len(inter_model_icc),
     note=(f"gate clause B on the ICC(2,k) point estimate, threshold "
           f"{ICC_THRESHOLD:.2f}, positive-case floor {MIN_POSITIVE_CASES}; "
           f"{len(MODELS)} model(s); leg modes: {_leg_note}"))

print("\nWHAT WAS PRODUCED")
print(f"  ensemble_scores.csv   {len(ensemble_scores):,} rows -- "
      f"{scored.shape[0]:,} response x item rows, {len(FACETS)} facets"
      + (f" + {len(EXTRA_ELICITATIONS)} separate-call rows" if EXTRA_ELICITATIONS else ""))
print(f"  inter_model_icc.csv   {len(inter_model_icc)} rows -- one per facet")
print(f"  raters: {len(MODELS)} ({', '.join(MODELS)})"
      + ("" if len(MODELS) >= 3 else "  <-- BELOW the registered minimum of three"))
print(f"  the scores were produced in: {_leg_note}")
print(f"  responses scored by every model: {N_ALL_LEGS:,} -- the sample behind "
      "every ICC above")
print("\nWHAT IS STILL NOT KNOWN")
print("  Clause B says the models agreed with each other. It does not say they")
print("  were right. Clause C (facet against its matching survey scale) has not")
print("  run yet, and clause A (agreement with human coders) is an addition to")
print("  the registration that runs only when HUMAN_GOLD is True. Where it does")
print("  not run it is reported as 'not run': never as False, which would record")
print("  every facet as having failed a test nobody applied.")
print("  A facet is gate-backed only where the clauses that ran all passed, and")
print("  that decision is made once, in the gate notebook, on these two files.")